## Bibliotecas

In [29]:
import os
from datetime import datetime
import pandas as pd
from pathlib import Path
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter

ROOT = Path().resolve()
OUTPUT  = ROOT / "output" 

## Baixando bases   

In [30]:
# baixando base ANS - Operadoras 

link_ativas = 'https://dadosabertos.ans.gov.br/FTP/PDA/operadoras_de_plano_de_saude_ativas/Relatorio_cadop.csv'
link_canceladas = 'https://dadosabertos.ans.gov.br/FTP/PDA/operadoras_de_plano_de_saude_canceladas/Relatorio_cadop_canceladas.csv'

In [31]:
df_ativas = pd.read_csv(link_ativas, sep=';', encoding='utf-8')
df_canceladas = pd.read_csv(link_canceladas, sep=';', encoding='utf-8')

In [32]:
df_ativas_1 = df_ativas.copy()
df_ativas_1['Data_Descredenciamento'] = ''
df_ativas_1['Motivo_do_Descredenciamento'] = ''

df_total = pd.concat([df_ativas_1, df_canceladas], ignore_index=True)


In [33]:
lst_relatorios = {
    'Relatorio Geral': 'df_total',
    'Relatorio Ativos': 'df_ativas',
    'Relatorio Cancelados': 'df_canceladas'
}

## Montando relatorio

In [34]:
df_para_exportar = df_total

#  configurações
arquivo_saida = "teste.xlsx"
nome_aba = "Base_Total"      
cabecalho = "Relatório ANS - Operadoras (Base Total)"  

COR_FONTE_CABECALHO_ARQUIVO = "00995D"
COR_FUNDO_CABECALHO_TABELA = "B1D34B"

vUsuario = 'lornelas'

vInicioTable = 3
vColorTable = vInicioTable+1
vCongela = f'A{vInicioTable+2}'
vAgora = datetime.now().strftime("%d/%m/%Y %H:%M:%S")

In [35]:
with pd.ExcelWriter(os.path.join(OUTPUT, arquivo_saida), engine="openpyxl") as writer:
    df_para_exportar.to_excel(writer, sheet_name=nome_aba, index=False, startrow=vInicioTable)
    
    ws = writer.sheets[nome_aba]
    
    ws["A1"] = cabecalho
    ws["A1"].font = Font(bold=True, size=14, color=COR_FONTE_CABECALHO_ARQUIVO)
    ws["A1"].alignment = Alignment(horizontal="left", vertical="center")

    ws["A2"] = f'Exportado por: {vUsuario} em {vAgora}'

 
    

    fill = PatternFill(fill_type="solid", fgColor=COR_FUNDO_CABECALHO_TABELA)
    fonte_negrito = Font(bold=True)
    
    # Aplicar preenchimento em todo o intervalo da linha 4
    for row in ws.iter_rows(min_row=vColorTable, max_row=vColorTable, min_col=1, max_col=len(df_para_exportar.columns)):
        for cell in row:
            cell.fill = fill
            cell.font = fonte_negrito
    
    ws.freeze_panes = vCongela

## Montando Relatorio Completo

In [36]:

# dicionário 
lst_relatorios = {
    "Relatorio Geral": df_total,
    "Relatorio Ativos": df_ativas,
    "Relatorio Cancelados": df_canceladas,
}

arquivo_saida = "teste_2.xlsx"

COR_FONTE_CABECALHO_ARQUIVO = "00995D"
COR_FUNDO_CABECALHO_TABELA = "B1D34B"

vUsuario = "lornelas"
vInicioTable = 3
vColorTable = vInicioTable + 1
vCongela = f"A{vInicioTable+2}"
vAgora = datetime.now().strftime("%d/%m/%Y %H:%M:%S")

def limpa_nome_aba(nome: str) -> str:
    # Excel: máximo 31 chars e não pode ter estes caracteres
    proibidos = r'[]:*?/\\'
    for ch in proibidos:
        nome = nome.replace(ch, "-")
    return nome[:31]

with pd.ExcelWriter(os.path.join(OUTPUT, arquivo_saida), engine="openpyxl") as writer:

    for nome_relatorio, df_para_exportar in lst_relatorios.items():
        nome_aba = limpa_nome_aba(nome_relatorio)
        cabecalho = f"Relatório ANS - Operadoras ({nome_relatorio})"

        df_para_exportar.to_excel(writer, sheet_name=nome_aba, index=False, startrow=vInicioTable)
        ws = writer.sheets[nome_aba]

        # Cabeçalho do arquivo
        ws["A1"] = cabecalho
        ws["A1"].font = Font(bold=True, size=14, color=COR_FONTE_CABECALHO_ARQUIVO)
        ws["A1"].alignment = Alignment(horizontal="left", vertical="center")

        ws["A2"] = f"Exportado por: {vUsuario} em {vAgora}"

        # Cabeçalho da tabela (REGISTRO, CNPJ, etc)
        fill = PatternFill(fill_type="solid", fgColor=COR_FUNDO_CABECALHO_TABELA)
        fonte_negrito = Font(bold=True)

        for row in ws.iter_rows(
            min_row=vColorTable,
            max_row=vColorTable,
            min_col=1,
            max_col=len(df_para_exportar.columns),
        ):
            for cell in row:
                cell.fill = fill
                cell.font = fonte_negrito

        ws.freeze_panes = vCongela